# Advanced: Multi-Strategy Uniform Temporal Noise with PAMOLA.CORE

**Goal**: Master all 3 temporal noise strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Boundary-constrained noise with bidirectional shifts
- **Strategy 2**: Pattern preservation (weekends + time-of-day)
- **Strategy 3**: Special date preservation with forward-only shifts
- Calculate temporal privacy metrics
- Analyze shift distributions and patterns
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of temporal privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/noise/
        └── 02_uniform_temporal_noise_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.noise.uniform_temporal_op import UniformTemporalNoiseOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 6 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, unique counts)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic patient visit data with temporal patterns
    base_date = pd.Timestamp('2024-01-01')
    
    # Create diverse datetime scenarios
    visit_datetimes = []
    admission_datetimes = []
    discharge_datetimes = []
    
    for i in range(n_records):
        # Visit datetime (8 AM - 6 PM, Mon-Fri mostly)
        day_offset = np.random.randint(0, 365)
        is_weekend = np.random.random() < 0.15  # 15% weekend visits
        
        if is_weekend:
            # Weekend: Saturday or Sunday
            day_offset = (day_offset // 7) * 7 + np.random.choice([5, 6])
        else:
            # Weekday: Monday to Friday
            day_offset = (day_offset // 7) * 7 + np.random.choice([0, 1, 2, 3, 4])
        
        visit_dt = base_date + timedelta(
            days=day_offset,
            hours=np.random.randint(8, 18),
            minutes=np.random.randint(0, 60)
        )
        visit_datetimes.append(visit_dt)
        
        # Admission (for some patients, 1-3 days after visit)
        if np.random.random() < 0.3:  # 30% get admitted
            admission_dt = visit_dt + timedelta(
                days=np.random.randint(1, 4),
                hours=np.random.randint(0, 24)
            )
            admission_datetimes.append(admission_dt)
            
            # Discharge (2-7 days after admission)
            discharge_dt = admission_dt + timedelta(
                days=np.random.randint(2, 8),
                hours=np.random.randint(0, 24)
            )
            discharge_datetimes.append(discharge_dt)
        else:
            admission_datetimes.append(pd.NaT)
            discharge_datetimes.append(pd.NaT)
    
    df = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(1, n_records + 1)],
        'visit_datetime': visit_datetimes,
        'admission_datetime': admission_datetimes,
        'discharge_datetime': discharge_datetimes,
        'department': np.random.choice(['Cardiology', 'Neurology', 'Orthopedics', 'Pediatrics', 'Emergency'], n_records),
        'severity': np.random.choice(['Low', 'Medium', 'High', 'Critical'], n_records)
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

# Convert datetime columns
for col in ['snapshot_date', 'visit_datetime', 'admission_datetime', 'discharge_datetime']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:25s} ({dtype_str:20s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        non_null = df[col].notna().sum()
        if non_null > 0:
            min_val = df[col].min()
            max_val = df[col].max()
            span = (max_val - min_val).days
            print(f"  {col:25s} ({dtype_str:20s}): {non_null} non-null, span={span} days")
        else:
            print(f"  {col:25s} ({dtype_str:20s}): all null")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Prepare Special Dates Configuration

**How to use:**
- Run to check/create special dates configuration
- Uses existing file if found
- Creates example holidays if missing

**What you'll see:**
- File status (✓ found or 🔧 created)
- File location path
- Structure summary (format version, date count)
- Sample dates listed

**Note:** Used by Strategy 3 for special date preservation

In [ ]:
# Define special dates path (in examples directory)
examples_dir = project_root / 'examples'
special_dates_path = examples_dir / 'data_examples' / 'special_dates.json'

# Check if special dates file already exists
if special_dates_path.exists():
    print(f"✓ Found existing special dates: {special_dates_path}")
    print(f"📂 Using existing file for Strategy 3\n")
    
    # Load and display info
    with open(special_dates_path, 'r') as f:
        existing_dates = json.load(f)
    
    print("📊 Existing File Info:")
    print(f"  Format:   v{existing_dates.get('format_version', 'N/A')}")
    print(f"  Type:     {existing_dates.get('date_type', 'N/A')}")
    print(f"  Count:    {len(existing_dates.get('dates', []))} dates")
    
    print("\n💡 To use different dates, replace or rename this file.")
    print("=" * 80)

else:
    print("⚠️  Special dates file not found!")
    print("🔧 Creating example special dates configuration...\n")
    print("=" * 80)

    # Define special dates structure
    special_dates_config = {
        "format_version": "1.0",
        "date_type": "holidays_and_events",
        "description": "Important dates to preserve during temporal noise",
        "dates": [
            "2024-01-01",  # New Year's Day
            "2024-07-04",  # Independence Day
            "2024-12-25",  # Christmas
            "2024-11-28",  # Thanksgiving
            "2024-03-17",  # St. Patrick's Day
            "2024-02-14",  # Valentine's Day
        ]
    }

    # Save to file (with error handling)
    try:
        special_dates_path.parent.mkdir(parents=True, exist_ok=True)
        with open(special_dates_path, 'w') as f:
            json.dump(special_dates_config, f, indent=2)
        print(f"✓ Example special dates created: {special_dates_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {special_dates_path} (file may be open)")
        print(f"   Data exists in memory only")

    print(f"\n📊 Statistics:")
    print(f"  Format version: {special_dates_config['format_version']}")
    print(f"  Date type: {special_dates_config['date_type']}")
    print(f"  Total dates: {len(special_dates_config['dates'])}")

    print(f"\n📅 Special Dates:")
    for date in special_dates_config['dates']:
        dt = pd.Timestamp(date)
        print(f"  {date} ({dt.day_name()})")

    print("\n💡 You can edit this file to match your important dates!")
    print("=" * 80)

## Step 4: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field": "snapshot_date"` → YOUR datetime column
   - `"strategy2_field": "snapshot_date"` → YOUR datetime column
   - `"strategy3_field": "snapshot_date"` → YOUR datetime column
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Date ranges per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "snapshot_date",      # Boundary-constrained noise
    "strategy2_field": "snapshot_date",  # Pattern preservation
    "strategy3_field": "snapshot_date"   # Special date preservation
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    
    # Convert to datetime if needed
    if not pd.api.types.is_datetime64_any_dtype(df[field_name]):
        df[field_name] = pd.to_datetime(df[field_name])
    
    # Display field info
    non_null = df[field_name].notna().sum()
    if non_null > 0:
        min_val = df[field_name].min()
        max_val = df[field_name].max()
        span = (max_val - min_val).days
        print(f"  ✓ {strategy:20s}: '{field_name}' ({non_null} values, {span} day span)")
    else:
        print(f"  ✓ {strategy:20s}: '{field_name}' (all null)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="temporal_advanced_001",
    task_type="multi_strategy_temporal",
    description="Multi-strategy temporal noise processing",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Boundary-Constrained Bidirectional Noise

**How to use:**
- Review pre-configured parameters
- Run to execute boundary-constrained strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → generate → transform → constrain → round → finalize
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `noise_range_days=14`: Random shift of ±14 days
- `direction='both'`: Allow forward or backward shifts
- `min_datetime='2024-01-01'`: Enforce minimum boundary
- `max_datetime='2024-12-31'`: Enforce maximum boundary
- `output_granularity='hour'`: Round to nearest hour

**Note:** Shifts clamped to boundaries when they exceed date range

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: BOUNDARY-CONSTRAINED BIDIRECTIONAL NOISE")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Boundary-constrained",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = UniformTemporalNoiseOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',  # Keep original + add new field
    
    # Strategy configuration
    noise_range_days=14,  # ±14 days random shift
    direction='both',     # Allow both forward and backward
    
    # Boundary constraints
    min_datetime='2024-01-01',
    max_datetime='2024-12-31',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_bounded",
    output_format='csv',
    output_granularity='hour',
    
    # Randomness
    use_secure_random=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_boundary_constrained',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_boundary_constrained' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_bounded"
    
    # Convert to datetime
    result_df_s1[field_s1] = pd.to_datetime(result_df_s1[field_s1])
    result_df_s1[output_field_s1] = pd.to_datetime(result_df_s1[output_field_s1])
    
    # Calculate shifts
    shifts = (result_df_s1[output_field_s1] - result_df_s1[field_s1]).dt.total_seconds() / 86400
    print(f"\n📊 Shift Analysis: mean={shifts.mean():.2f} days, std={shifts.std():.2f} days")

## STRATEGY 2: Pattern Preservation (Weekends + Time-of-Day)

**How to use:**
- Preserves weekend/weekday status and time-of-day patterns
- Run to execute pattern preservation strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → generate → preserve patterns → metrics → visualize → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `noise_range_days=10`: Random shift of ±10 days
- `preserve_weekends=True`: Maintain Mon-Fri vs Sat-Sun status
- `preserve_time_of_day=True`: Keep original time component
- `direction='both'`: Allow forward or backward shifts
- `output_granularity='minute'`: Round to nearest minute

**Note:** Best for preserving business patterns and daily cycles

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: PATTERN PRESERVATION (WEEKENDS + TIME-OF-DAY)")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Pattern preservation",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = UniformTemporalNoiseOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    
    # Strategy configuration
    noise_range_days=10,
    direction='both',
    
    # Pattern preservation
    preserve_weekends=True,
    preserve_time_of_day=True,
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_preserved",
    output_format='csv',
    output_granularity='minute',
    
    # Randomness
    use_secure_random=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_pattern_preservation',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_pattern_preservation' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_preserved"
    
    # Convert to datetime
    result_df_s2[field_s2] = pd.to_datetime(result_df_s2[field_s2])
    result_df_s2[output_field_s2] = pd.to_datetime(result_df_s2[output_field_s2])
    
    # Analyze pattern preservation
    non_null = result_df_s2[field_s2].notna()
    if non_null.any():
        orig_weekends = result_df_s2.loc[non_null, field_s2].dt.dayofweek.isin([5, 6])
        new_weekends = result_df_s2.loc[non_null, output_field_s2].dt.dayofweek.isin([5, 6])
        weekend_preserved = (orig_weekends == new_weekends).sum()
        print(f"\n📊 Weekend preservation: {weekend_preserved}/{non_null.sum()} records ({weekend_preserved/non_null.sum()*100:.1f}%)")

## STRATEGY 3: Special Date Preservation + Forward-Only

**How to use:**
- Preserves special dates (holidays) and only shifts forward
- Run to execute special date preservation strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → generate → preserve dates → metrics → visualize → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `noise_range_days=7`: Random shift of 0-7 days
- `direction='forward'`: Only future shifts (no past)
- `preserve_special_dates=True`: Keep holidays unchanged
- `special_dates`: Load from special_dates.json
- `output_granularity='day'`: Round to nearest day

**Note:** Optimal for scenarios requiring monotonic temporal progression

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: SPECIAL DATE PRESERVATION + FORWARD-ONLY")
print("=" * 80 + "\n")

# Load special dates
special_dates = None
if special_dates_path.exists():
    with open(special_dates_path, 'r') as f:
        special_dates_data = json.load(f)
        special_dates = special_dates_data.get('dates', [])
        print(f"✓ Loaded {len(special_dates)} special dates from {special_dates_path.name}")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Special date preservation",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = UniformTemporalNoiseOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    
    # Strategy configuration
    noise_range_days=7,
    direction='forward',  # Only shift to future
    
    # Special date preservation
    preserve_special_dates=True,
    special_dates=special_dates,
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_forward",
    output_format='csv',
    output_granularity='day',
    
    # Randomness
    use_secure_random=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_special_dates',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_special_dates' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_forward"
    
    # Convert to datetime
    result_df_s3[field_s3] = pd.to_datetime(result_df_s3[field_s3])
    result_df_s3[output_field_s3] = pd.to_datetime(result_df_s3[output_field_s3])
    
    # Analyze forward-only shifts
    non_null = result_df_s3[field_s3].notna()
    if non_null.any():
        shifts = (result_df_s3.loc[non_null, output_field_s3] - result_df_s3.loc[non_null, field_s3]).dt.total_seconds() / 86400
        forward_count = (shifts > 0).sum()
        zero_count = (shifts == 0).sum()
        print(f"\n📊 Direction analysis: {forward_count} forward, {zero_count} unchanged (100% forward-only)")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Shift direction breakdown per strategy
- Pattern preservation effectiveness

**Note:** Fastest strategy isn't always best - consider privacy requirements

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print("\n📊 Strategy Characteristics:")
print(f"  Strategy 1: Boundary-constrained, ±14 days, both directions")
print(f"  Strategy 2: Pattern preservation (weekends + time), ±10 days")
print(f"  Strategy 3: Special dates + forward-only, 0-7 days")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Noise config, actual shifts, direction breakdown, preservation stats
2. **Visualizations**: Temporal distribution and shift pattern charts

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_boundary_constrained', 'Strategy 1: Boundary-Constrained'),
    ('strategy2_pattern_preservation', 'Strategy 2: Pattern Preservation'),
    ('strategy3_special_dates', 'Strategy 3: Special Date Preservation')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                
                # Display noise configuration
                if 'noise_range_config' in data:
                    config = data['noise_range_config']
                    print(f"   Range: {config.get('days', 0)} days ({config.get('total_seconds', 0)} seconds)")
                
                # Display actual shifts
                if 'actual_shifts' in data:
                    shifts = data['actual_shifts']
                    print(f"   Mean shift: {shifts.get('mean_days', 0):.2f} days")
                    print(f"   Max shift: {shifts.get('max_days', 0):.2f} days")
                
                # Display direction breakdown
                if 'shift_direction' in data:
                    direction = data['shift_direction']
                    print(f"   Forward: {direction.get('forward_count', 0)}, Backward: {direction.get('backward_count', 0)}, Zero: {direction.get('zero_count', 0)}")
                
                # Display preservation settings
                if 'preservation' in data:
                    pres = data['preservation']
                    print(f"   Weekends preserved: {pres.get('weekends_preserved', False)}")
                    print(f"   Time-of-day preserved: {pres.get('time_of_day_preserved', False)}")
                    print(f"   Special dates preserved: {pres.get('special_dates_preserved', False)}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Calculate Temporal Privacy Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates temporal uncertainty improvements

**What you'll see:**
- Original data: date ranges, span analysis
- Anonymized data: expanded ranges per strategy
- Temporal uncertainty ratios (higher is better)

**Privacy targets:**
- Uncertainty ≥ 1.5x: Basic temporal protection
- Uncertainty ≥ 2.0x: Strong temporal protection
- No exact timestamp matches: Ideal

**Note:** Requires result_df_s{N} from strategy execution cells

In [ ]:
print("\n" + "=" * 80)
print("🔒 TEMPORAL PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_temporal_uncertainty(original: pd.Series, anonymized: pd.Series) -> dict:
    """Calculate temporal uncertainty metrics."""
    # Filter non-null values
    non_null = original.notna() & anonymized.notna()
    orig_clean = original[non_null]
    anon_clean = anonymized[non_null]
    
    if len(orig_clean) == 0:
        return {'error': 'no non-null values'}
    
    # Calculate ranges
    orig_range = (orig_clean.max() - orig_clean.min()).total_seconds() / 86400
    anon_range = (anon_clean.max() - anon_clean.min()).total_seconds() / 86400
    
    # Calculate unique timestamps
    orig_unique = orig_clean.nunique()
    anon_unique = anon_clean.nunique()
    
    return {
        'original_range_days': orig_range,
        'anonymized_range_days': anon_range,
        'range_expansion': anon_range / orig_range if orig_range > 0 else 0,
        'original_unique': orig_unique,
        'anonymized_unique': anon_unique,
        'uniqueness_ratio': anon_unique / orig_unique if orig_unique > 0 else 0
    }

# Check if FIELD_CONFIG exists (strategies completed)
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
    
    # Strategy 1 metrics
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        print("📊 STRATEGY 1: BOUNDARY-CONSTRAINED")
        output_field_s1 = f"{field_s1}_bounded"
        if output_field_s1 in result_df_s1.columns:
            metrics_s1 = calculate_temporal_uncertainty(
                result_df_s1[field_s1],
                result_df_s1[output_field_s1]
            )
            print(f"   Original range: {metrics_s1['original_range_days']:.1f} days")
            print(f"   Anonymized range: {metrics_s1['anonymized_range_days']:.1f} days")
            print(f"   Range expansion: {metrics_s1['range_expansion']:.2f}x")
    
    # Strategy 2 metrics
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        print("\n📊 STRATEGY 2: PATTERN PRESERVATION")
        output_field_s2 = f"{field_s2}_preserved"
        if output_field_s2 in result_df_s2.columns:
            metrics_s2 = calculate_temporal_uncertainty(
                result_df_s2[field_s2],
                result_df_s2[output_field_s2]
            )
            print(f"   Original range: {metrics_s2.get('original_range_days', 0):.1f} days")
            print(f"   Anonymized range: {metrics_s2.get('anonymized_range_days', 0):.1f} days")
            print(f"   Range expansion: {metrics_s2.get('range_expansion', 0):.2f}x")
    
    # Strategy 3 metrics
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        print("\n📊 STRATEGY 3: SPECIAL DATE PRESERVATION")
        output_field_s3 = f"{field_s3}_forward"
        if output_field_s3 in result_df_s3.columns:
            metrics_s3 = calculate_temporal_uncertainty(
                result_df_s3[field_s3],
                result_df_s3[output_field_s3]
            )
            print(f"   Original range: {metrics_s3.get('original_range_days', 0):.1f} days")
            print(f"   Anonymized range: {metrics_s3.get('anonymized_range_days', 0):.1f} days")
            print(f"   Range expansion: {metrics_s3.get('range_expansion', 0):.2f}x")
    
    print("\n🎯 Higher range expansion = better temporal privacy protection")
        
except NameError:
    print("⚠️  Run Step 4 (Setup Environment) first!")

## Step 8: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports anonymized datasets for each strategy

**What you'll see (per strategy):**
- Available columns list
- Export path
- Preview (first 5 rows)
- Statistics (shift ranges)

**Output structure:**
```
advanced_output/
├── strategy1_boundary_constrained/
│   └── temporal_anonymized_data.csv
├── strategy2_pattern_preservation/
│   └── temporal_anonymized_data.csv
└── strategy3_special_dates/
    └── temporal_anonymized_data.csv
```

**Note:** Files include both original and anonymized fields for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
except NameError:
    print("❌ Run Step 4 first!")
    raise

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Boundary-Constrained
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: BOUNDARY-CONSTRAINED")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_boundary_constrained'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_bounded"
    
    if output_col_s1 in result_df_s1.columns:
        export_cols_s1 = ['patient_id', field_s1, output_col_s1, 'department', 'severity']
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        # Save
        output_path_s1 = s1_dir / 'temporal_anonymized_data.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        # Preview
        print(f"\n📊 Preview:")
        print(final_df_s1.head())
        
        # Statistics
        shifts = (result_df_s1[output_col_s1] - result_df_s1[field_s1]).dt.total_seconds() / 86400
        print(f"\n📈 Shift range: {shifts.min():.2f} to {shifts.max():.2f} days")
    else:
        print(f"\n❌ Column '{output_col_s1}' not found")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Pattern Preservation
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: PATTERN PRESERVATION")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_pattern_preservation'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s2 = f"{field_s2}_preserved"
    
    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['patient_id', field_s2, output_col_s2, 'department', 'severity']
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        # Save
        output_path_s2 = s2_dir / 'temporal_anonymized_data.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        # Preview
        print(f"\n📊 Preview:")
        print(final_df_s2.head())
    else:
        print(f"\n❌ Column '{output_col_s2}' not found")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Special Date Preservation
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: SPECIAL DATE PRESERVATION")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_special_dates'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s3 = f"{field_s3}_forward"
    
    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['patient_id', field_s3, output_col_s3, 'department', 'severity']
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        # Save
        output_path_s3 = s3_dir / 'temporal_anonymized_data.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        # Preview
        print(f"\n📊 Preview:")
        print(final_df_s3.head())
    else:
        print(f"\n❌ Column '{output_col_s3}' not found")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 temporal noise strategies implemented and compared
- ✅ Temporal privacy metrics calculated
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Boundary constraints ensure temporal values stay within valid ranges
- Pattern preservation maintains critical business logic (weekends, daily cycles)
- Special date preservation protects important temporal landmarks
- Direction control provides flexibility for different privacy scenarios

**Strategy recommendations:**
- **Use Strategy 1** when: You need controlled temporal shifts within specific date boundaries
- **Use Strategy 2** when: Business patterns (weekends, time-of-day) must be preserved
- **Use Strategy 3** when: Important dates (holidays, events) must remain unchanged and monotonic progression required

**Next steps:**
- Test with your own datetime datasets
- Tune noise ranges for optimal privacy-utility balance
- Combine with other anonymization operations
- Deploy to production environment

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)